Santiago Vieira Ceballos
Sara Franco Taborda 


# Taller ingeniería de datos

El problema que se desea resolver haciendo un modelo predictivo es el siguiente:

*En la industria de telecomunicaciones, la **retención de clientes** es uno de los mayores desafíos para las empresas, ya que la pérdida de clientes (churn) afecta directamente los ingresos y la sostenibilidad del negocio. En este contexto, la empresa que proporciona servicios de telecomunicaciones a través del dataset tiene como objetivo reducir la **tasa de cancelación de sus servicios**. En particular, la variable **Churn Label** identifica a aquellos clientes que han decidido abandonar el servicio. Para abordar este problema, es crucial predecir con antelación qué clientes podrían cancelar su suscripción, para implementar estrategias de retención específicas. Este análisis predictivo no solo ayudaría a identificar a los clientes en riesgo de churn, sino que también podría revelar factores determinantes como la **razón de cancelación**, **tipos de contrato**, **método de pago**, **tasa de uso** y otras variables que influyen en la decisión de los clientes. Por lo tanto, realizar un análisis predictivo basado en estas variables permitirá a la empresa tomar medidas proactivas para mejorar su **estrategia de fidelización** y, en última instancia, reducir el churn y aumentar la **rentabilidad** a largo plazo.*

El equipo de análisis cuenta con un dataset cuyo diccionario de datos se puede consultar en [IBM](https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113).

## Carga e identificación de problemas de los datos

Abra el archivo de datos en Excel o bloc de notas, y revise sí hay algo particular con los datos. Este paso es importante para evitar problemas posteriores al cargar los datos.

Cargue en `Pandas` el archivo dado, que contiene un dataset cuya información puede consultar en [IBM](https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113).

Indique cuántas variables y registros tiene el dataset, y asegúrese que el tipo de dato de cada variable sea el esperado.

Reporte las estadísticas descriptivas de las variables numéricas y categóricas.

Identifique si el dataframe tiene datos duplicados.

Identifique si hay variables con datos nulos.

Reporte los hallazgos en un celda de texto.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path().resolve().parent / "data" / "raw"
data_file = "Telco_customer_churn.csv"
data_path = DATA_DIR / data_file
df = pd.read_csv(data_path,na_values=[' ']) 
df.info()

In [ ]:
# Estadisticas Numericas 
df.describe().T


In [ ]:
#Estadisticas Categóricas
df.describe(include="object").T


In [ ]:
#Suma de datos Duplicados en el dataset 
df.duplicated().sum()

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(5)

In [ ]:
# las varianles se leyeron de forma adecuada 
# no encontramos datos duplicados en el dataset
# Churn reason tiene 5174 datos nulos 
# Total Charges tiene 11 datos nulos
# Country e state tiene solo un valor osea que es constante 



Descarte las variables que considere irrelevantes para su modelo (p.e. claves priamrias, variables con valores únicos, variables redundantes, entre otros).

Gestione los datos duplicados y los datos nulos.

In [ ]:
df = df.set_index('CustomerID')

In [ ]:

df = df.drop(columns=['Count', 'Country', 'State', 'Lat Long'])

In [ ]:
df =  df.drop_duplicates()

In [ ]:
df = df.drop(columns=['Churn Reason'])
df.info()


In [ ]:
df = df.dropna(subset=['Total Charges'])

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(5)

## Análisis de variables categóricas

Haga un análisis de las variables categóricas no descartadas, e identifique:

- Variables nominales.
- Variables ordinales.
- Variables con alta cardinalidad.
- Variables que no deberían incluirse en la matriz de características.

Reporte los resultados del análisis en una celda de texto.

In [ ]:
# entregar posibles resultados de las varibles tipo object 
var_categoricas = df.select_dtypes(include="object").columns 
for var in var_categoricas: 
    print(f'{var} : {df[var].unique()}')

In [ ]:
#df = df.drop(columns=['City'])
var_categoricas = df.select_dtypes(include="object").columns 

In [ ]:
import matplotlib.pyplot as plt

#analsis bivairado entre variables categoricas y variable objetivo: churn label
for var in var_categoricas:
    tabla = pd.crosstab(df[var], df['Churn Label'], normalize='index')
    tabla.plot(kind='bar', stacked=True)
    plt.title(var)
    plt.show()

In [ ]:
# Se decide excluir del modelo las variables categóricas Gender, PhoneService y MultipleLines,
# ya que el análisis exploratorio muestra que, independientemente de la categoría que presenten,
# no evidencian una relación significativa con la variable objetivo (Churn Label).
# Por esta razón, no se consideran variables adecuadas para ser utilizadas como predictores en el modelo.

In [ ]:
df = df.drop(columns=['Gender','Phone Service', 'Multiple Lines'])

## Análisis de variables cuantitativas

Haga un análisis de las variables cuantitativas no descartadas, e identifique:

- Variables que son aproximadamente normales.
- Variables con datos atípicos.
- Variables que no deberían incluirse en la matriz de características.

Reporte los resultados del análisis en una celda de texto.

In [ ]:
num_vars = df.select_dtypes(include='number').columns
for var in num_vars:
    plt.figure()
    df.boxplot(by='Churn Label', column=var)
    plt.title(var)
    plt.show()

In [ ]:
#Se eliminaran las variables numericas zip code latitud y longitud
# debido a que no son varibles representativas a las variables predictorias.

In [ ]:
df = df.drop(columns=['Zip Code', 'Latitude', 'Longitude'])

In [ ]:
# Histogramas de las variables numéricas 
df.select_dtypes('number').hist(bins=20, figsize=(15,10), grid=False, layout=(2,3))

In [ ]:
import matplotlib.pyplot as plt

num_vars = df.select_dtypes(include='number').columns

for var in num_vars:
    plt.figure()              
    df.boxplot(column=var, grid=False)
    plt.title(f'Boxplot de {var}')
    plt.show()

In [ ]:
# Ninguno de los anteriores Boxplots muestra la presencia de outliers significativos en las variables numéricas del dataset.

In [ ]:
# analisaremos por medio de asimetria y kurtosis la normalidad de las variables.
for num in df.select_dtypes(include=['number']).columns:
    print(f"Asimetría de {num}: {df[num].skew():.3f} \n")

In [ ]:
for num in df.select_dtypes(include=['number']).columns:
    print(f"Kurtosis de {num}: {df[num].kurt():.3f} \n")

In [ ]:
# La variable total charges tiene una kurtosis cercana a 0 lo cual es la que mas se aproxma a ser una variable normal.

## Entrenamiento del modelo

Entrene un modelo de árbol de decisión que prediga la variable **Churn Value**.

Procese las variables de acuerdo con los análisis de variables cuantitativas y categóricas hechos previamente. Este procesamiento debe empaquetarse usando `ColumnTransformer`, para que los datos de entrenamiento y prueba sean procesados por separado.

Reporte los `scores` de entrenamiento y prueba.

Reporte las características con las que **realmente** fue entrenado el modelo, es decir, las resultantes del preprocesamiento.

In [ ]:
#Primero dividir los datos de entrenamiento y prueba, luego se hace el preprocesamiento
from sklearn.model_selection import train_test_split

y = df['Churn Value']
X = df.drop(['Churn Label', 'Churn Value'], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1, train_size=0.7)
print(f'Tamaño del conjunto de entrenamiento es: {X_train.shape}')
print(f'Tamaño del conjunto de prueba es: {X_test.shape}')

In [ ]:
from sklearn.preprocessing import (OneHotEncoder,OrdinalEncoder,PowerTransformer,StandardScaler)

ss = StandardScaler() # Para preprocesar age y hours-per-week
pt = PowerTransformer() # Para preprocesar fnlwgt
orden = ['Month-to-month', 'One year', 'Two year']
ore = OrdinalEncoder(categories=[orden], dtype='int') # Para preprocesar education
ohe = OneHotEncoder(sparse_output=False, drop='if_binary') # workclass, marital-status, occupation, relationship, race y sex

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(transformers=[
    ('num_prep', pt, ['Total Charges']),#normalizando
    ('num_prep2', ss, ['Tenure Months', 'Monthly Charges', 'Churn Score','CLTV']), # estandarizando
    ('cod_education', ore, ['Contract']), # Volviendo ordinal , asignar niveles
    ('cod_oh', ohe, ['Senior Citizen', 'Partner', 'Dependents', 'Internet Service', 'Online Security', 'Online Backup', 
                     'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Paperless Billing', 'Payment Method' ])
    ], remainder='drop') # Las demás columnas se descartan

In [ ]:
preprocessor

In [ ]:
var_categoricas = X_train.select_dtypes(include="object").columns 
var_categoricas

In [ ]:
# PREPARACION DE LOS DATOS DE ENTRENAMIENTO Y PRUEBA PARA EL MODELO
preprocessor.fit(X_train)
X_train_prep = preprocessor.transform(X_train)
X_test_prep = preprocessor.transform(X_test)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(
    max_depth=5,
    random_state=1
    )

tree.fit(X_train_prep, y_train)

In [ ]:
print(f'Accuracy en el conjunto de entrenamiento: {tree.score(X_train_prep, y_train)}')
print(f'Accuracy en el conjunto de prueba: {tree.score(X_test_prep, y_test)}')

Aqui puedo observar las caracteristicas que fueron entrenadas en mi modelo 
|

In [ ]:
print(preprocessor.get_feature_names_out())

In [ ]:
print(
    'Número de características con las que fue entrenado el modelo: ', 
    preprocessor.get_feature_names_out().shape[0]
    )